# Patch mixing off on the NYU final model (3 seeds)

**Configuration** (identical to the PM run of notebook 10, 'NYU ablations re-based on the NYU final model', which is seed 42 here):
- the notebook 09 configuration plus `frame_pe=False, photo_rng_fix=False` (option A, i.e. the notebook 06 8-block arm), plus `use_patch_mix=False`;
- that is: 8 fine-tuned blocks, layer decay 0.9, critic off, linear output, rescale [1, 1.5], crop 476×630, standard PE, 20k iterations, batch 8.
- Seeds 42 (already trained in notebook 10, cached here), 43 and 44 (≈ 3.3 h each).
- Cityscapes is unchanged: its final model keeps patch mixing, because PM was never measured on Cityscapes and no Cityscapes run is planned.

**Comparison.** The 3 seeds are compared with the 3 seeds of the NYU final model (notebook 06 8-block arm, `results/seeds/nc8_s{42,43,44}`, patch mixing on). This gives the PM ablation on 3 + 3 seeds.

**Outcome (user, 2026-11-11):** patch mixing off improves depth but lowers our per-image mIoU below SwinMTL's 58.14, so patch mixing stays on in the NYU final model.

## Setup

***NOTE:*** The DINOv2 and DPT code is used straight from the local repositories `./dinov2` and `./DPT`.

In [1]:
import os, sys
if os.path.basename(os.getcwd()) == "experiments":
    os.chdir("..")
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

In [ ]:
from mtl_final import *
from repro_common import PAPER   # the papers' reference numbers, defined once (checked against the PDFs)

configure(results_root=ROOT / "results")

CONFIG = load_json(FINAL_DIR / "config_final.json")
FINAL_TAG = "log" if CONFIG["depth_param"] == "log" else "base"
NYU_NOPM = dict(frame_pe=False, photo_rng_fix=False, use_patch_mix=False)   # option A + patch mixing off
TAG = f"{FINAL_TAG}-nofpe-nopatchmix"                                       # same folders as the PM run of notebook 10 (seed 42)
SEEDS = [42, 43, 44]
print("configuration:", CONFIG, "| overrides:", NYU_NOPM, "| tag:", TAG)

sigma: {'rmse': 0.003, 'miou': 0.2, 'abs_rel_cv': 0.01, 'source': 'notebook 06 (raw: rmse=0.001669, miou=0.1389, abs_rel_cv=0.005997)'}
configuration: {'blocks': 8, 'use_critic': False, 'frame_pe': True, 'photo_rng_fix': True, 'depth_param': 'linear', 'nyu_scale_aug': [1.0, 1.5], 'nyu_crop': None, 'nyu_rotate_deg': 0.0} | overrides: {'frame_pe': False, 'photo_rng_fix': False, 'use_patch_mix': False} | tag: base-nofpe-nopatchmix


## Runs
Seed 42 is the PM run of notebook 10 ('NYU ablations re-based on the NYU final model'): it is found in `ckpt/final/nyu_base-nofpe-nopatchmix_s42` and only loaded. Seeds 43 and 44 are trained. Each run resumes from `last.pt` after an interruption.

In [3]:
R_NOPM = {}
for seed in SEEDS:
    R_NOPM[seed] = train_eval("nyu", seed, TAG, CONFIG, **NYU_NOPM)
    r = R_NOPM[seed]
    print(f"seed {seed}: RMSE {r['nyu_depth']['rmse']:.4f}, AbsRel {r['nyu_depth']['abs_rel']:.4f}, δ1 {r['nyu_depth']['d1']:.4f}, "
          f"mIoU {r['nyu_seg']['mIoU']:.2f} (per-image {r['nyu_seg_swinmtl']['mIoU']:.2f}), log bias {r['log_bias']:+.4f}")

=== nyu base-nofpe-nopatchmix seed 42: /home/chrisjf/Documents/ml4cv/ckpt/final/nyu_base-nofpe-nopatchmix_s42
nyu_final: already trained
seed 42: RMSE 0.4021, AbsRel 0.1150, δ1 0.8798, mIoU 58.57 (per-image 57.63), log bias +0.0227
=== nyu base-nofpe-nopatchmix seed 43: /home/chrisjf/Documents/ml4cv/ckpt/final/nyu_base-nofpe-nopatchmix_s43
nyu_final: already trained
seed 43: RMSE 0.4041, AbsRel 0.1145, δ1 0.8807, mIoU 58.70 (per-image 57.95), log bias +0.0192
=== nyu base-nofpe-nopatchmix seed 44: /home/chrisjf/Documents/ml4cv/ckpt/final/nyu_base-nofpe-nopatchmix_s44
nyu_final: already trained
seed 44: RMSE 0.4046, AbsRel 0.1153, δ1 0.8792, mIoU 57.53 (per-image 57.27), log bias +0.0186


## Patch mixing on vs off, 3 seeds each
- **On:** the previous NYU final model, notebook 06 8-block arm, `results/seeds/nc8_s{seed}`.
- **Off:** this notebook.

The table gives mean ± std over seeds and the z-score of the decision rule (difference of means / σ of notebook 09, floors included). z > 0 means patch mixing off is better.

In [4]:
METRICS = [("RMSE", "nyu_depth", "rmse", True), ("AbsRel", "nyu_depth", "abs_rel", True), ("δ1", "nyu_depth", "d1", False),
           ("RMSE flip (TTA)", "nyu_depth_flip", "rmse", True), ("mIoU", "nyu_seg", "mIoU", False), ("mIoU per-image (SwinMTL)", "nyu_seg_swinmtl", "mIoU", False)]
R_ON = {s: load_eval(ROOT / "results" / "seeds" / f"nc8_s{s}", "nyu_partial_ft8") for s in SEEDS}
rows, per_run = [], []
for name, key, metric, lower in METRICS:
    on = np.array([R_ON[s][key][metric] for s in SEEDS]); off = np.array([R_NOPM[s][key][metric] for s in SEEDS])
    sig = {"RMSE": SIGMA["rmse"], "mIoU": SIGMA["miou"]}.get(name)
    z = ((on.mean() - off.mean()) if lower else (off.mean() - on.mean())) / sig if sig else float("nan")
    rows.append(dict(metric=name, PM_on=f"{on.mean():.4f} ± {on.std(ddof=1):.4f}", PM_off=f"{off.mean():.4f} ± {off.std(ddof=1):.4f}",
                     delta=off.mean() - on.mean(), z_rule=z))
for s in SEEDS:
    for arm, R in (("PM on", R_ON[s]), ("PM off", R_NOPM[s])):
        per_run.append(dict(arm=arm, seed=s, RMSE=R["nyu_depth"]["rmse"], AbsRel=R["nyu_depth"]["abs_rel"], d1=R["nyu_depth"]["d1"],
                            mIoU=R["nyu_seg"]["mIoU"], mIoU_per_image=R["nyu_seg_swinmtl"]["mIoU"]))
pm_table = pd.DataFrame(rows)
out_dir = ROOT / "results" / "compare"
out_dir.mkdir(parents=True, exist_ok=True)
pm_table.to_csv(out_dir / "pm_seeds_nyu.csv", index=False)
pd.DataFrame(per_run).to_csv(out_dir / "pm_seeds_nyu_per_run.csv", index=False)
display(pm_table.style.format(precision=4))
display(pd.DataFrame(per_run).style.format(precision=4))
j = judge("PM (3 seeds)", "nyu", {"nyu_depth": {"rmse": np.mean([R_ON[s]["nyu_depth"]["rmse"] for s in SEEDS])}, "nyu_seg": {"mIoU": np.mean([R_ON[s]["nyu_seg"]["mIoU"] for s in SEEDS])}},
          {"nyu_depth": {"rmse": np.mean([R_NOPM[s]["nyu_depth"]["rmse"] for s in SEEDS])}, "nyu_seg": {"mIoU": np.mean([R_NOPM[s]["nyu_seg"]["mIoU"] for s in SEEDS])}})
print(f"decision rule on the 3-seed means: z_depth {j['z_depth']:+.2f}, z_seg {j['z_seg']:+.2f}, better={j['better']}, harm={j['harm']}")
print(f"targets: SwinMTL NYU RMSE {PAPER["swinmtl_tab3"]["SwinMTL"][0]:.3f}, mIoU {PAPER["swinmtl_tab3"]["SwinMTL"][1]} (as reported, SwinMTL's per-image metric) | DINOv2 + DPT RMSE {PAPER["dinov2_tab11_vits14"]["nyu"][2]} (frozen, ~24k frames)")

,metric,PM_on,PM_off,delta,z_rule
0,RMSE,0.4197 ± 0.0013,0.4036 ± 0.0013,-0.0161,5.3821
1,AbsRel,0.1204 ± 0.0010,0.1149 ± 0.0004,-0.0055,nan
2,δ1,0.8652 ± 0.0014,0.8799 ± 0.0008,0.0147,nan
3,RMSE flip (TTA),0.4136 ± 0.0013,0.3983 ± 0.0013,-0.0152,nan
4,mIoU,58.0957 ± 0.1956,58.2664 ± 0.6374,0.1708,0.8538
5,mIoU per-image (SwinMTL),58.4299 ± 0.2186,57.6163 ± 0.3399,-0.8136,nan


,arm,seed,RMSE,AbsRel,d1,mIoU,mIoU_per_image
0,PM on,42,0.4205,0.1215,0.8637,58.2490,58.5486
1,PM off,42,0.4021,0.1150,0.8798,58.5685,57.6325
2,PM on,43,0.4205,0.1198,0.8663,58.1626,58.5634
3,PM off,43,0.4041,0.1145,0.8807,58.6966,57.9478
4,PM on,44,0.4183,0.1199,0.8656,57.8754,58.1777
5,PM off,44,0.4046,0.1153,0.8792,57.5341,57.2686


decision rule on the 3-seed means: z_depth +5.38, z_seg +0.85, better=True, harm=False
targets: SwinMTL NYU RMSE 0.518, mIoU 58.14 (as reported, SwinMTL's per-image metric) | DINOv2 + DPT RMSE 0.356 (frozen, ~24k frames)
